In [1]:
from mlp import Linear, BatchNorm1D, Tanh
import torch 
import torch.nn.functional as F
from contextensor import ContextTorchTensor, TensorSplit


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/vicentearjona/Documents/LLM_practice/.venv/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()

# Generate torch tensors from name list

In [2]:
list_to_tensor = ContextTorchTensor(context=4)
list_to_tensor.open(file='names.txt')
X, Y = list_to_tensor.get_tensors()


# Train/validation/test splits

In [3]:
tsplit = TensorSplit(train_size=0.8 , test_size=0.1)
xtrain, xval, xtest, ytrain, yval, ytest = tsplit.split(xs=X, ys=Y)

# Initialize parameters 

In [4]:
dim_emb = 3 # embedding dimension
batch_size = 32 # number examples extracted for each batch. 
dim_hidden = 100 # dimensionality hidden layers. Number of neurons of the hidden layers 
g = 2147483647
vocab_size = list_to_tensor.vocab_size
context = list_to_tensor.context
num_iterations = 200000 
lr = 0.1

## Definition of the tensors

In [5]:
C = torch.randn(size=(vocab_size, dim_emb), generator=torch.Generator().manual_seed(g)) # embedding tensor 
# Definition of the layer elements 
layers = [Linear(fan_in=dim_emb * context, fan_out=dim_hidden,generator=g), Tanh(),
          Linear(fan_in=dim_hidden, fan_out=dim_hidden, generator=g),       Tanh(),
          Linear(fan_in=dim_hidden, fan_out=dim_hidden, generator=g),       Tanh(),
          Linear(fan_in=dim_hidden, fan_out=vocab_size, generator=g,bias=False)]    

## Solving initialization issues

In [ ]:
with torch.no_grad():
    layers[-1].weight *= 0.01
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.bias *= 0.01 
            layer.weight *= 5/3

## Parameters 

In [ ]:
parameters = [C] + [p for layer in layers for p in layer.parameters()]
for p in parameters:
    p.requires_grad = True

In [ ]:
print(f'number of parameters = {sum([p.nelement() for p in parameters])}')

# Forward and backward pass 

In [ ]:
# For loop to update the parameters of the NN via gradient descent 
loss_train = [] # object where we store the loss of each iteration 
up_to_data = [] # object where we store the update to date data of each iteration 
for i in range(num_iterations):
# Select minibatch from the complete set 
    ixs = torch.randint(low=0, high=xtrain.shape[0], size=(batch_size,))
    xtr = xtrain[ixs]
    ytr = ytrain[ixs]
# Forward pass. Returns loss
    emb = C[xtr]
    X = emb.view(-1, emb.shape[1] * emb.shape[2])
# Iteration over layers 
    for layer in layers:
        X = layer(X)
# Append loss 
    loss_i = F.cross_entropy(input=X, target=ytr)
    loss_train.append(loss_i.log10().item())
# Backward pass. Updates grads
    for layer in layers:
        layer.out.retain_grad() 
    for p in parameters: 
        p.grad = None 
    loss_i.backward() # fill grad attributes. Gradient descent for the loss 
# update pass. Recalculates params
    lr = lr if i < int(0.75 * num_iterations) else lr / 100
    for p in parameters:
        p.data += -lr * p.grad
        # Store up to date pass. For each iteration, we store the update to data of each parameter 
        with torch.no_grad():
            up_to_data.append(((lr * p.grad).std() / p.data.std()).log10().item())
    if i % 10000 == 0:
        print(f"Iteration:{i}")


In [6]:
with torch.no_grad(): 
    def split_loss(split:str) -> str: 
        X, Y = {
            'train': [xtrain, ytrain], 
            'test': [xtest, ytest], 
            'val': [xval, yval]
        }[split]
        emb = C[X]
        X = emb.view(-1, emb.shape[1] * emb.shape[2])
        # Iteration over layers 
        for layer in layers:
            X = layer(X)
        # Append loss 
        return f'{split}: loss = {F.cross_entropy(input=X, target=Y)}'

In [ ]:
for layer in layers:
    layer.train = False 
print(split_loss('train'))
print(split_loss('test'))
print(split_loss('val'))

## Sample from the model 

In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10)
out = [] 
for _ in range(10): # Number of words to be sample 
    xstart = [0] * context # initial character. Starting character * context 
    while True: 
        # embeding 
        emb = C[torch.tensor([xstart])]
        x = emb.view(-1, emb.shape[1]*emb.shape[2])
        for layer in layers:
            x = layer(x) 
        # after iterating over all the layers, the resulting output is the logit tensor
        probs = F.softmax(x,dim=1) # we compute the probabilities 
        ix = torch.multinomial(probs, num_samples=1, generator=g).item() # we extract the next character 
        xstart = xstart[1:] + [ix] 
        out.append(list_to_tensor.itos[ix])
        if ix == 0: 
            break
print(''.join(i for i in out)) # decode and print the generated word 

# Applying BATCHNORM after each linear layer 

## Definition of the tensors

In [32]:
g = 2147483647
C = torch.randn(size=(vocab_size, dim_emb), generator=torch.Generator().manual_seed(g)) # embedding tensor 
# Definition of the layer elements 
layers = [Linear(fan_in=dim_emb * context, fan_out=dim_hidden,generator=g, bias=False), BatchNorm1D(dim_hidden), Tanh(),
          Linear(fan_in=dim_hidden, fan_out=dim_hidden, generator=g, bias=False),       BatchNorm1D(dim_hidden), Tanh(),
          Linear(fan_in=dim_hidden, fan_out=dim_hidden, generator=g, bias=False),       BatchNorm1D(dim_hidden), Tanh(),
          Linear(fan_in=dim_hidden, fan_out=dim_hidden, generator=g, bias=False),       BatchNorm1D(dim_hidden), Tanh(),
          Linear(fan_in=dim_hidden, fan_out=dim_hidden, generator=g, bias=False),       BatchNorm1D(dim_hidden), Tanh(),
          Linear(fan_in=dim_hidden, fan_out=vocab_size, generator=g, bias=False),       BatchNorm1D(vocab_size)]    

## Solving initialization issues

In [33]:
with torch.no_grad():
    layers[-1].bngain *= 0.1
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.weight *= 5/3

## Parameters 

In [34]:
parameters = [C] + [p for layer in layers for p in layer.parameters()]
for p in parameters:
    p.requires_grad = True

In [35]:
print(f'number of parameters = {sum([p.nelement() for p in parameters])}')

number of parameters = 45035


# Forward and backward pass 

In [36]:
# For loop to update the parameters of the NN via gradient descent 
loss_train = [] # object where we store the loss of each iteration 
up_to_data = [] # object where we store the update to date data of each iteration 
for i in range(num_iterations):
# Select minibatch from the complete set 
    ixs = torch.randint(low=0, high=xtrain.shape[0], size=(batch_size,))
    xtr = xtrain[ixs]
    ytr = ytrain[ixs]
# Forward pass. Returns loss
    emb = C[xtr]
    X = emb.view(-1, emb.shape[1] * emb.shape[2])
# Iteration over layers 
    for layer in layers:
        X = layer(X)
# Append loss 
    loss_i = F.cross_entropy(input=X, target=ytr)
    loss_train.append(loss_i.log10().item())
# Backward pass. Updates grads
    for layer in layers:
        layer.out.retain_grad() 
    for p in parameters: 
        p.grad = None 
    loss_i.backward() # fill grad attributes. Gradient descent for the loss 
# update pass. Recalculates params
    lr = lr if i < int(0.75 * num_iterations) else lr / 100
    for p in parameters:
        p.data += -lr * p.grad
        # Store up to date pass. For each iteration, we store the update to data of each parameter 
        with torch.no_grad():
            up_to_data.append(((lr * p.grad).std() / p.data.std()).log10().item())
    if i % 10000 == 0:
        print(f"Iteration:{i}. Loss: {loss_i}")


Iteration:0. Loss: 3.2928242683410645
Iteration:10000. Loss: 2.1915347576141357
Iteration:20000. Loss: 2.407473087310791
Iteration:30000. Loss: 1.850020170211792
Iteration:40000. Loss: 1.8537356853485107
Iteration:50000. Loss: 1.9724974632263184
Iteration:60000. Loss: 1.871999740600586
Iteration:70000. Loss: 1.9831575155258179
Iteration:80000. Loss: 1.8980756998062134
Iteration:90000. Loss: 2.492969512939453
Iteration:100000. Loss: 2.4572691917419434
Iteration:110000. Loss: 1.658350944519043
Iteration:120000. Loss: 1.8202435970306396
Iteration:130000. Loss: 2.221214771270752
Iteration:140000. Loss: 2.1767683029174805
Iteration:150000. Loss: 2.3165199756622314
Iteration:160000. Loss: 1.770214557647705
Iteration:170000. Loss: 2.091531753540039
Iteration:180000. Loss: 2.544339418411255
Iteration:190000. Loss: 2.183779716491699


### Saturation

In [37]:

for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out
    print('layer %d (%10s): mean %+.2f, std %.2f, saturated: %.2f%%' % (i, layer.__class__.__name__, t.mean(), t.std(), (t.abs() > 0.97).float().mean()*100))

layer 2 (      Tanh): mean -0.03, std 0.73, saturated: 19.75%
layer 5 (      Tanh): mean -0.04, std 0.77, saturated: 23.88%
layer 8 (      Tanh): mean -0.01, std 0.79, saturated: 26.06%
layer 11 (      Tanh): mean -0.01, std 0.79, saturated: 24.81%
layer 14 (      Tanh): mean +0.02, std 0.80, saturated: 26.59%


### Gradient statistics 

In [38]:
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out.grad
    print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))

layer 2 (      Tanh): mean -0.000000, std 3.466183e-03
layer 5 (      Tanh): mean -0.000000, std 3.770089e-03
layer 8 (      Tanh): mean +0.000000, std 3.729702e-03
layer 11 (      Tanh): mean +0.000000, std 3.456346e-03
layer 14 (      Tanh): mean -0.000000, std 3.587136e-03


### Gradient to data ratio

In [39]:
# visualize histograms
for i,p in enumerate(parameters):
  t = p.grad
  if p.ndim == 2:
    print('weight %10s | mean %+f | std %e | grad:data ratio %e' % (tuple(p.shape), t.mean(), t.std(), t.std() / p.std()))
   

weight    (27, 3) | mean -0.000000 | std 1.868393e-02 | grad:data ratio 1.275084e-02
weight  (12, 100) | mean +0.000174 | std 8.643490e-03 | grad:data ratio 1.411329e-02
weight (100, 100) | mean -0.000007 | std 4.590780e-03 | grad:data ratio 1.709517e-02
weight (100, 100) | mean -0.000002 | std 5.149056e-03 | grad:data ratio 1.855373e-02
weight (100, 100) | mean -0.000019 | std 4.907442e-03 | grad:data ratio 1.781551e-02
weight (100, 100) | mean -0.000041 | std 4.417800e-03 | grad:data ratio 1.690505e-02
weight  (100, 27) | mean +0.000039 | std 6.829680e-03 | grad:data ratio 1.778700e-02


## Train/test/validation loss 

In [40]:
for layer in layers: 
    layer.train = False    
print(split_loss('train'))
print(split_loss('val'))
print(split_loss('test'))

train: loss = 2.026536703109741
val: loss = 2.0864620208740234
test: loss = 2.0855472087860107


## Sample from the model

In [41]:
xs = xtrain[torch.randint(low=0,high=xtrain.shape[0],size=[32,])]
emb = C[xs]
xs = emb.view(-1,emb.shape[1]*emb.shape[2])
for layer in layers: 
    xs = layer(xs)
xs -= xs.max(dim=1,keepdim=True).values 
F.softmax(xs,dim=1)


tensor([[3.1188e-03, 1.8245e-01, 7.9761e-03, 1.2836e-03, 3.8809e-03, 3.2059e-01,
         2.9190e-03, 2.5097e-03, 1.0212e-01, 5.3697e-02, 6.6885e-03, 1.7361e-02,
         1.1479e-02, 1.3299e-02, 7.5023e-03, 1.2467e-01, 3.7786e-03, 5.8140e-04,
         6.0369e-02, 1.3090e-02, 5.5272e-03, 1.1083e-02, 1.4164e-02, 6.3560e-03,
         1.1116e-03, 2.1227e-02, 1.1593e-03],
        [2.5411e-01, 1.5184e-01, 1.2566e-03, 3.7081e-03, 6.6120e-02, 6.0428e-02,
         1.0619e-03, 4.7485e-02, 7.9388e-04, 1.3610e-01, 2.8024e-03, 3.6350e-03,
         3.2701e-02, 6.7047e-03, 7.1224e-02, 1.7312e-02, 7.8175e-05, 7.0549e-04,
         3.9885e-03, 3.3242e-02, 1.6619e-02, 8.5397e-04, 2.5825e-03, 1.5566e-04,
         3.8944e-04, 7.4220e-02, 9.8839e-03],
        [5.9256e-01, 6.4245e-02, 1.5432e-04, 1.3931e-03, 9.4735e-03, 1.7224e-01,
         4.5034e-05, 7.3559e-03, 5.1206e-04, 6.2044e-02, 1.2132e-04, 1.0531e-03,
         1.5255e-03, 6.7251e-04, 4.9293e-02, 7.3206e-03, 2.4075e-05, 1.2193e-04,
         3.8921e-

In [42]:
g = torch.Generator().manual_seed(2147483647 + 20)
out = [] 
for _ in range(10): # Number of words to be sample 
    xstart = [0] * context # initial character. Starting character * context 
    while True: 
        # embeding 
        emb = C[torch.tensor([xstart])]
        x = emb.view(-1, emb.shape[1]*emb.shape[2])
        for layer in layers:
            x = layer(x) 
        # after iterating over all the layers, the resulting output is the logit tensor
        probs = F.softmax(x,dim=1) # we compute the probabilities 
        ix = torch.multinomial(probs, num_samples=1, generator=g).item() # we extract the next character 
        xstart = xstart[1:] + [ix] 
        out.append(list_to_tensor.itos[ix])
        if ix == 0: 
            break
print(''.join(i for i in out)) # decode and print the generated word 

zeily*adesrien*britt*halson*kenra*yaneymen*abby*arniel*ashba*mollianvra*


# Practice with tensor reshaping

In [73]:
e1 = torch.randint(0,9,(4,6,5))
e1

tensor([[[6, 0, 8, 0, 7],
         [0, 0, 0, 4, 1],
         [1, 6, 7, 0, 7],
         [3, 4, 2, 0, 6],
         [2, 1, 7, 8, 7],
         [3, 8, 4, 8, 0]],

        [[3, 5, 8, 0, 0],
         [8, 2, 1, 8, 0],
         [4, 5, 1, 5, 2],
         [2, 7, 1, 4, 7],
         [8, 1, 7, 1, 4],
         [6, 8, 0, 4, 1]],

        [[1, 6, 0, 5, 1],
         [5, 0, 2, 2, 7],
         [4, 1, 8, 8, 2],
         [7, 0, 8, 4, 4],
         [6, 2, 0, 6, 6],
         [1, 4, 0, 1, 1]],

        [[4, 1, 3, 0, 4],
         [7, 7, 1, 2, 3],
         [8, 5, 6, 7, 0],
         [5, 6, 4, 0, 0],
         [1, 3, 1, 5, 8],
         [1, 0, 0, 0, 1]]])

In [72]:
e1.view(4,6//3,-1)

tensor([[[1, 5, 3, 2, 4, 2, 5, 8, 3, 5, 5, 1, 3, 7, 2],
         [3, 1, 4, 2, 2, 3, 3, 7, 2, 0, 7, 5, 8, 6, 5]],

        [[7, 3, 3, 2, 0, 2, 7, 6, 5, 5, 0, 5, 4, 8, 4],
         [1, 8, 8, 6, 4, 1, 4, 3, 1, 4, 3, 7, 5, 6, 4]],

        [[1, 4, 3, 7, 8, 0, 6, 3, 0, 0, 8, 5, 4, 6, 7],
         [6, 3, 8, 8, 7, 4, 2, 8, 6, 0, 5, 2, 6, 7, 6]],

        [[7, 7, 7, 0, 2, 0, 3, 0, 6, 1, 7, 4, 4, 2, 5],
         [0, 5, 0, 2, 2, 6, 3, 6, 4, 8, 8, 5, 3, 5, 0]]])

In [74]:
e1[:,::2,:], e1[:,1::2,:]

(tensor([[[6, 0, 8, 0, 7],
          [1, 6, 7, 0, 7],
          [2, 1, 7, 8, 7]],
 
         [[3, 5, 8, 0, 0],
          [4, 5, 1, 5, 2],
          [8, 1, 7, 1, 4]],
 
         [[1, 6, 0, 5, 1],
          [4, 1, 8, 8, 2],
          [6, 2, 0, 6, 6]],
 
         [[4, 1, 3, 0, 4],
          [8, 5, 6, 7, 0],
          [1, 3, 1, 5, 8]]]),
 tensor([[[0, 0, 0, 4, 1],
          [3, 4, 2, 0, 6],
          [3, 8, 4, 8, 0]],
 
         [[8, 2, 1, 8, 0],
          [2, 7, 1, 4, 7],
          [6, 8, 0, 4, 1]],
 
         [[5, 0, 2, 2, 7],
          [7, 0, 8, 4, 4],
          [1, 4, 0, 1, 1]],
 
         [[7, 7, 1, 2, 3],
          [5, 6, 4, 0, 0],
          [1, 0, 0, 0, 1]]]))